# Analysis Pipeline — Food-101 / SHViT vs Baselines

Runs the full analysis after all 6 models are trained. Everything stays on Colab's local SSD (`/content/`) so you can inspect outputs in the file browser. At the end, a single zip is created which you can download or copy to Drive.

**Before running:** `Runtime → Change runtime type → A100 GPU` (T4 also works, just slower for the eval step).

### What this notebook does
1. Mounts Drive (read-only access to checkpoints + dataset)
2. Copies dataset to local SSD for fast reads
3. Pulls the latest analysis scripts from your repo
4. Runs `eda.py`, `evaluate_all.py`, `make_figures.py`, `error_analysis.py`, `demo.py` in order
5. Inspects results and zips everything for download

## 0. GPU check

In [ ]:
import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

## 1. Mount Drive (read-only access to checkpoints + dataset source)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Everything this notebook reads and writes lives under one root directory:
BASE_DIR = '/content/drive/MyDrive/CV_Research_Paper_Food101'

# Source paths on Drive (trained models + dataset, from Stages 2 & 3)
DRIVE_DATA      = f'{BASE_DIR}/food101_data'
DRIVE_BASELINES = f'{BASE_DIR}/Stage 2: baseline models'
DRIVE_SHVIT_FT  = f'{BASE_DIR}/Stage 3: fine-tuning SHViT'

# Analysis working paths — kept on local SSD, still under CV_Research_Paper_Food101
LOCAL_ROOT        = '/content/CV_Research_Paper_Food101/analysis'
LOCAL_DATA        = '/content/CV_Research_Paper_Food101/food101_data'
LOCAL_CHECKPOINTS = f'{LOCAL_ROOT}/checkpoints'
LOCAL_RESULTS     = f'{LOCAL_ROOT}/results'
LOCAL_FIGURES     = f'{LOCAL_ROOT}/figures'
LOCAL_EDA         = f'{LOCAL_ROOT}/eda_outputs'
LOCAL_ERROR       = f'{LOCAL_ROOT}/error_analysis_outputs'
LOCAL_DEMO        = f'{LOCAL_ROOT}/demo_output'

import os
for p in [LOCAL_ROOT, LOCAL_CHECKPOINTS, LOCAL_RESULTS, LOCAL_FIGURES,
          LOCAL_EDA, LOCAL_ERROR, LOCAL_DEMO]:
    os.makedirs(p, exist_ok=True)
print('Analysis root:', LOCAL_ROOT)

## 2. Copy dataset to local SSD

If `/content/CV_Research_Paper_Food101/food101_data` already exists from an earlier session, this skips the copy.

In [ ]:
import shutil, time

if not os.path.exists(f'{LOCAL_DATA}/food-101/images'):
    print('Copying dataset from Drive to local SSD...')
    t0 = time.time(); shutil.copytree(DRIVE_DATA, LOCAL_DATA); print(f'Done in {time.time()-t0:.0f}s')
else:
    classes = len(os.listdir(f'{LOCAL_DATA}/food-101/images'))
    print(f'Already copied — {classes} class folders present')

# Make sure the Tip-Adapter Zhou split file is present in the local dataset.
SPLIT_PATH = f'{LOCAL_DATA}/food-101/split_zhou_Food101.json'
if not os.path.exists(SPLIT_PATH):
    !pip install -q gdown
    !gdown "https://drive.google.com/uc?id=1QK0tGi096I0Ba6kggatX1ee6dJFIcEJl" -O "{SPLIT_PATH}"
print('Zhou split present:', os.path.exists(SPLIT_PATH))

## 3. Copy trained checkpoints from Drive to local

Symlinks would also work, but copies are safer and only ~300 MB total.

In [ ]:
# Layout matches what evaluate_all.py expects:
#   <repo>/Stage 2: baseline models/<model>/best.pth
#   <repo>/Stage 3: fine-tuning SHViT/<model>/best.pth
STAGE2_LOCAL = f'{LOCAL_CHECKPOINTS}/Stage 2: baseline models'
STAGE3_LOCAL = f'{LOCAL_CHECKPOINTS}/Stage 3: fine-tuning SHViT'
os.makedirs(STAGE2_LOCAL, exist_ok=True)
os.makedirs(STAGE3_LOCAL, exist_ok=True)

for model in ['resnet50', 'mobilenet_v2']:
    src_dir = f'{DRIVE_BASELINES}/{model}'
    dst_dir = f'{STAGE2_LOCAL}/{model}'
    if os.path.isdir(src_dir) and not os.path.isdir(dst_dir):
        shutil.copytree(src_dir, dst_dir)
        print(f'copied {model}: {os.path.getsize(f"{dst_dir}/best.pth")/1e6:.1f} MB')
    elif os.path.isdir(dst_dir):
        print(f'{model}: already present')
    else:
        print(f'WARNING: {model} not found at {src_dir}')

for model in ['shvit_s1', 'shvit_s2', 'shvit_s3', 'shvit_s4']:
    src_dir = f'{DRIVE_SHVIT_FT}/{model}'
    dst_dir = f'{STAGE3_LOCAL}/{model}'
    if os.path.isdir(src_dir) and not os.path.isdir(dst_dir):
        shutil.copytree(src_dir, dst_dir)
        print(f'copied {model}: {os.path.getsize(f"{dst_dir}/best.pth")/1e6:.1f} MB')
    elif os.path.isdir(dst_dir):
        print(f'{model}: already present')
    else:
        print(f'WARNING: {model} not found at {src_dir}')

## 4. Clone repos and install dependencies

Pulls the latest analysis scripts from your branch. If the repo already exists, `git pull` updates it.

In [ ]:
REPO_DIR  = '/content/Vision_Project_spring_26'
REPO_URL  = 'https://github.com/saif-farid-tech/Vision_Project_spring_26.git'
BRANCH    = 'Vision_Project_spring_26_Food101'
SHVIT_DIR = '/content/SHViT'

import os, shutil

if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)

!git clone -b {BRANCH} {REPO_URL} {REPO_DIR}

if not os.path.isdir(SHVIT_DIR):
    !git clone https://github.com/ysj9909/SHViT.git {SHVIT_DIR}

# Copy analysis scripts to /content for clean invocation
for fname in ['eda.py', 'evaluate_all.py', 'make_figures.py',
              'error_analysis.py', 'demo.py',
              'augmentation.py', 'metrics.py', 'splits.py']:
    src = f'{REPO_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/{fname}')
        print(f'copied {fname}')
    else:
        print(f'MISSING: {fname} (push it to your branch first)')

# Copy the tip_datasets package (Tip-Adapter preprocessing + Zhou split loader).
if os.path.isdir('/content/tip_datasets'):
    shutil.rmtree('/content/tip_datasets')
shutil.copytree(f'{REPO_DIR}/tip_datasets', '/content/tip_datasets')
print('copied tip_datasets/')

In [ ]:
# Dependencies: timm pinned, fvcore for GFLOPs, seaborn for heatmaps
!pip install -q timm==0.5.4 --no-deps
!pip install -q einops==0.4.1 easydict fvcore seaborn gdown
print('Deps installed.')

## 5. EDA — sample grid, class distribution, image sizes

Quick (~2 min). No GPU needed.

In [ ]:
!python /content/eda.py \
    --data-root  {LOCAL_DATA} \
    --output-dir {LOCAL_EDA}

print('\nEDA outputs:')
for f in sorted(os.listdir(LOCAL_EDA)):
    sz = os.path.getsize(f'{LOCAL_EDA}/{f}') / 1024
    print(f'  {f}  ({sz:.0f} KB)')

## 6. Full evaluation + benchmarks on the test set

This is the big one — runs all 6 models on the official 25,250-image test set, plus parameter / FLOPs / throughput / latency benchmarks. ~30 min on A100.

In [ ]:
!python /content/evaluate_all.py \
    --checkpoints-dir {LOCAL_CHECKPOINTS} \
    --shvit-dir       {SHVIT_DIR} \
    --data-root       {LOCAL_DATA} \
    --output-dir      {LOCAL_RESULTS}

print('\nResults outputs:')
for root, dirs, files in os.walk(LOCAL_RESULTS):
    for f in files:
        full = os.path.join(root, f)
        rel  = os.path.relpath(full, LOCAL_RESULTS)
        sz   = os.path.getsize(full) / 1024
        print(f'  {rel}  ({sz:.0f} KB)')

In [ ]:
# Quick peek at the headline table
table_path = f'{LOCAL_RESULTS}/results_table.md'
if os.path.exists(table_path):
    with open(table_path) as f:
        print(f.read())

## 7. Generate report figures

Reads training logs (from the checkpoint dirs) + `results.json` files. Produces the speed-vs-accuracy headline figure, training curves, train loss, accuracy bars.

In [ ]:
!python /content/make_figures.py \
    --results-dir {LOCAL_RESULTS} \
    --logs-dir    {LOCAL_CHECKPOINTS} \
    --output-dir  {LOCAL_FIGURES}

print('\nFigures:')
for f in sorted(os.listdir(LOCAL_FIGURES)):
    sz = os.path.getsize(f'{LOCAL_FIGURES}/{f}') / 1024
    print(f'  {f}  ({sz:.0f} KB)')

In [ ]:
# Inline preview
from IPython.display import Image, display
for f in ['fig_speed_vs_accuracy.png', 'fig_training_curves.png',
          'fig_accuracy_bars.png', 'fig_train_loss.png']:
    p = f'{LOCAL_FIGURES}/{f}'
    if os.path.exists(p):
        print(f)
        display(Image(p, width=700))

## 8. Error analysis on SHViT-S4

Worst-15 categories, top-10 confused class pairs, confusion heatmap, misclassified image grid.

In [ ]:
S4_CKPT = f'{STAGE3_LOCAL}/shvit_s4/best.pth'

!python /content/error_analysis.py \
    --results-dir {LOCAL_RESULTS} \
    --data-root   {LOCAL_DATA} \
    --shvit-dir   {SHVIT_DIR} \
    --checkpoint  "{S4_CKPT}" \
    --output-dir  {LOCAL_ERROR}

print('\nError analysis outputs:')
for f in sorted(os.listdir(LOCAL_ERROR)):
    sz = os.path.getsize(f'{LOCAL_ERROR}/{f}') / 1024
    print(f'  {f}  ({sz:.0f} KB)')

In [ ]:
# Inline preview of error figures + summary text
for f in ['confusion_top20.png', 'misclassified_grid.png', 'worst_15_categories.png']:
    p = f'{LOCAL_ERROR}/{f}'
    if os.path.exists(p):
        print(f)
        display(Image(p, width=700))

summary = f'{LOCAL_ERROR}/error_analysis_summary.txt'
if os.path.exists(summary):
    with open(summary) as f:
        print(f.read())

## 9. Demo on a single image

Smoke test for `demo.py`. Pick any test image — the path below grabs the first ceviche image.

Edit `DEMO_IMAGE` to try other foods.

In [ ]:
import glob
DEMO_IMAGE = sorted(glob.glob(f'{LOCAL_DATA}/food-101/images/sushi/*.jpg'))[0]
print('Using image:', DEMO_IMAGE)

DEMO_OUT = f'{LOCAL_DEMO}/demo_output.png'

!python /content/demo.py \
    --image       "{DEMO_IMAGE}" \
    --checkpoint  "{S4_CKPT}" \
    --shvit-dir   {SHVIT_DIR} \
    --data-root   {LOCAL_DATA} \
    --output      "{DEMO_OUT}"

if os.path.exists(DEMO_OUT):
    display(Image(DEMO_OUT, width=600))

## 10. Bundle everything for download

Zips the entire `/content/analysis/` folder so you can download it from the Colab file browser (right-click `analysis.zip` → Download). The zip excludes the local checkpoint copies (those are still on Drive).

In [ ]:
ZIP_PATH = f'{LOCAL_ROOT}/CV_Research_Paper_Food101_analysis.zip'
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# Zip results, figures, EDA, error analysis, demo — skip checkpoints (large + already on Drive)
!cd {LOCAL_ROOT} && zip -rq {ZIP_PATH} \
    results figures eda_outputs error_analysis_outputs demo_output

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'Bundle ready: {ZIP_PATH}  ({size_mb:.1f} MB)')
print('Right-click CV_Research_Paper_Food101_analysis.zip in the file browser to download.')

In [ ]:
# Copy the analysis bundle to Drive, under "Stage 4: Benchmarking and Demo/".
STAGE4_DRIVE = f'{BASE_DIR}/Stage 4: Benchmarking and Demo'
os.makedirs(STAGE4_DRIVE, exist_ok=True)
DRIVE_BUNDLE = f'{STAGE4_DRIVE}/CV_Research_Paper_Food101_analysis.zip'
shutil.copy(ZIP_PATH, DRIVE_BUNDLE)
print(f'Also saved to: {DRIVE_BUNDLE}')